1. Read the dataset
2. Drop the columns which are unique for all users like IDs (2.5 points)
3. Distinguish the feature and target set (2.5 points)
4. Divide the data set into Train and test sets
5. Normalize the train and test data (2.5 points)
6. Initialize & build the model (10 points)
7. Optimize the model (5 points)
8. Predict the results using 0.5 as a threshold (5 points)
9. Print the Accuracy score and confusion matrix (2.5 points)


In [1]:
import keras

/opt/conda/lib/python3.6/site-packages/h5py/__init__.py:36: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


### Question
Read the dataset<br>
Drop the columns which are unique for all users like IDs (2.5 points)

In [2]:
import pandas as pd
import numpy as np

In [44]:
dataset = pd.read_csv('bank.csv')

In [45]:
dataset.head().T

,0,1,2,3,4
RowNumber,1,2,3,4,5
CustomerId,15634602,15647311,15619304,15701354,15737888
Surname,Hargrave,Hill,Onio,Boni,Mitchell
CreditScore,619,608,502,699,850
Geography,France,Spain,France,France,Spain
Gender,Female,Female,Female,Female,Female
Age,42,41,42,39,43
Tenure,2,1,8,1,2
Balance,0,83807.9,159661,0,125511
NumOfProducts,1,1,3,2,1


In [46]:
dataset.shape

(10000, 14)

In [47]:
X = dataset.iloc[:,3:13].values # Credit Score through Estimated Salary
y = dataset.iloc[:, 13].values # Exited
print(dataset.shape)
print(X.shape)
print(y.shape)

(10000, 14)
(10000, 10)
(10000,)


In [48]:
dataset.isna().sum()

RowNumber          0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64

### No Null Values. The df contains 10000 rows

### Question
Distinguish the feature and target set (2.5 points)

Already done above<br>
X is features and Y is target (Exited)

In [49]:
# Encoding categorical (string based) data. Country: there are 3 options: France, Spain and Germany
# This will convert those strings into scalar values for analysis
print(X[:8,1], '... will now become: ')
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
label_X_country_encoder = LabelEncoder()
X[:,1] = label_X_country_encoder.fit_transform(X[:,1])
print(X[:8,1])


['France' 'Spain' 'France' 'France' 'Spain' 'Spain' 'France' 'Germany'] ... will now become: 
[0 2 0 0 2 2 0 1]


In [50]:
# We will do the same thing for gender. this will be binary in this dataset
print(X[:6,2], '... will now become: ')
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
label_X_gender_encoder = LabelEncoder()
X[:,2] = label_X_gender_encoder.fit_transform(X[:,2])
print(X[:6,2])

['Female' 'Female' 'Female' 'Female' 'Female' 'Male'] ... will now become: 
[0 0 0 0 0 1]


In [51]:
# Converting the string features into their own dimensions. Gender doesn't matter here because its binary
countryhotencoder = OneHotEncoder(categorical_features = [1]) # 1 is the country column
X = countryhotencoder.fit_transform(X).toarray()

/opt/conda/lib/python3.6/site-packages/sklearn/preprocessing/_encoders.py:371: FutureWarning: The handling of integer data will change in version 0.22. Currently, the categories are determined based on the range [0, max(values)], while in the future they will be determined based on the unique values.
If you want the future behaviour and silence this warning, you can specify "categories='auto'".
In case you used a LabelEncoder before this OneHotEncoder to convert the categories to integers, then you can now use the OneHotEncoder directly.
  warnings.warn(msg, FutureWarning)
/opt/conda/lib/python3.6/site-packages/sklearn/preprocessing/_encoders.py:392: DeprecationWarning: The 'categorical_features' keyword is deprecated in version 0.20 and will be removed in 0.22. You can use the ColumnTransformer instead.
  "use the ColumnTransformer instead.", DeprecationWarning)


In [52]:
print(X.shape)

(10000, 12)


In [53]:
X = X[:,1:]

In [54]:
print(X.shape)

(10000, 11)


### Question
Divide the data set into Train and test sets

In [57]:
# Splitting the dataset into the Training and Testing set.
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 0)

### Question
Normalize the train and test data (2.5 points)

In [58]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [59]:
X_train.shape

(8000, 11)

In [60]:
X_train

array([[-0.5698444 ,  1.74309049,  0.16958176, ...,  0.64259497,
        -1.03227043,  1.10643166],
       [ 1.75486502, -0.57369368, -2.30455945, ...,  0.64259497,
         0.9687384 , -0.74866447],
       [-0.5698444 , -0.57369368, -1.19119591, ...,  0.64259497,
        -1.03227043,  1.48533467],
       ...,
       [-0.5698444 , -0.57369368,  0.9015152 , ...,  0.64259497,
        -1.03227043,  1.41231994],
       [-0.5698444 ,  1.74309049, -0.62420521, ...,  0.64259497,
         0.9687384 ,  0.84432121],
       [ 1.75486502, -0.57369368, -0.28401079, ...,  0.64259497,
        -1.03227043,  0.32472465]])

### Question
Initialize & build the model (10 points)

In [61]:
from keras.models import Sequential
from keras.layers import Dense

In [62]:
# Initializing the ANN
classifier = Sequential()

In [63]:
# This adds the input layer (by specifying input dimension) AND the first hidden layer (units)
classifier.add(Dense(activation = 'relu', input_dim = 11, units=6, kernel_initializer='uniform'))

In [64]:
# Adding the second hidden layer
# Notice that we do not need to specify input dim. 
classifier.add(Dense(activation = 'relu', units=6, kernel_initializer='uniform')) 

In [65]:
# Adding the output layer
# Notice that we do not need to specify input dim. 
# we have an output of 1 node, which is the the desired dimensions of our output (stay with the bank or not)
# We use the sigmoid because we want probability outcomes
classifier.add(Dense(activation = 'sigmoid', units=1, kernel_initializer='uniform')) 

In [66]:
classifier.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])

In [67]:
classifier.fit(X_train, y_train, batch_size=10, epochs=100)


Epoch 1/100
8000/8000 [==============================] - 1s 184us/step - loss: 0.5026 - acc: 0.7955
Epoch 2/100
8000/8000 [==============================] - 1s 121us/step - loss: 0.4137 - acc: 0.8210
Epoch 3/100
8000/8000 [==============================] - 1s 116us/step - loss: 0.4006 - acc: 0.8272
Epoch 4/100
8000/8000 [==============================] - 1s 109us/step - loss: 0.3909 - acc: 0.8299
Epoch 5/100
8000/8000 [==============================] - 1s 115us/step - loss: 0.3840 - acc: 0.8300
Epoch 6/100
8000/8000 [==============================] - 1s 118us/step - loss: 0.3788 - acc: 0.8337
Epoch 7/100
8000/8000 [==============================] - 1s 122us/step - loss: 0.3746 - acc: 0.8381
Epoch 8/100
8000/8000 [==============================] - 1s 120us/step - loss: 0.3714 - acc: 0.8441
Epoch 9/100
8000/8000 [==============================] - 1s 115us/step - loss: 0.3691 - acc: 0.8481
Epoch 10/100
8000/8000 [==============================] - 1s 115us/step - loss: 0.3665 - acc: 0.8477

8000/8000 [==============================] - 1s 117us/step - loss: 0.3366 - acc: 0.8615
Epoch 83/100
8000/8000 [==============================] - 1s 119us/step - loss: 0.3366 - acc: 0.8611
Epoch 84/100
8000/8000 [==============================] - 1s 116us/step - loss: 0.3368 - acc: 0.8647
Epoch 85/100
8000/8000 [==============================] - 1s 112us/step - loss: 0.3364 - acc: 0.8616
Epoch 86/100
8000/8000 [==============================] - 1s 114us/step - loss: 0.3360 - acc: 0.8605
Epoch 87/100
8000/8000 [==============================] - 1s 116us/step - loss: 0.3359 - acc: 0.8612
Epoch 88/100
8000/8000 [==============================] - 1s 116us/step - loss: 0.3365 - acc: 0.8622
Epoch 89/100
8000/8000 [==============================] - 1s 114us/step - loss: 0.3355 - acc: 0.8621
Epoch 90/100
8000/8000 [==============================] - 1s 114us/step - loss: 0.3350 - acc: 0.8626
Epoch 91/100
8000/8000 [==============================] - 1s 112us/step - loss: 0.3357 - acc: 0.8624
Epo

### Question
Predict the results using 0.5 as a threshold (5 points) and Print the Accuracy score and confusion matrix (2.5 points)

In [68]:
y_pred = classifier.predict(X_test)
print(y_pred)

[[0.2763576 ]
 [0.42877588]
 [0.1729469 ]
 ...
 [0.28422052]
 [0.1740847 ]
 [0.27100205]]


### Question
Predict the results using 0.5 as a threshold (5 points)

In [69]:
y_pred = (y_pred > 0.5)
print(y_pred)

[[False]
 [False]
 [False]
 ...
 [False]
 [False]
 [False]]


### Question
Print the Accuracy score and confusion matrix (2.5 points)

In [70]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[1475  120]
 [ 168  237]]


In [71]:
print (((cm[0][0]+cm[1][1])*100)/(cm[0][0]+cm[1][1]+cm[0][1]+cm[1][0]), '% of testing data was classified correctly')

85.6 % of testing data was classified correctly


### Question
Optimize the model (5 points)

In [73]:
#Using K-Fold cross validation to optimize the model
from sklearn.model_selection import StratifiedKFold
import numpy
# fix random seed for reproducibility
seed = 7
numpy.random.seed(seed)
# split into input (X) and output (Y) variables
X = X
Y = y
# define 10-fold cross validation test harness
kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)
cvscores = []
for train, test in kfold.split(X, Y):
# create model
    classifier = Sequential()
    classifier.add(Dense(activation = 'relu', input_dim = 11, units=6, kernel_initializer='uniform'))
    classifier.add(Dense(activation = 'relu', units=6, kernel_initializer='uniform'))     
    classifier.add(Dense(activation = 'sigmoid', units=1, kernel_initializer='uniform'))
# Compile model
    classifier.compile(optimizer='adam', loss = 'binary_crossentropy', metrics=['accuracy'])
# Fit the model
    classifier.fit(X[train], Y[train], epochs=100, batch_size=10, verbose=1)
# evaluate the model
    scores = model.evaluate(X[test], Y[test], verbose=0)
    print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
    cvscores.append(scores[1] * 100)
print("%.2f%% (+/- %.2f%%)" % (numpy.mean(cvscores), numpy.std(cvscores)))

Epoch 1/100
8999/8999 [==============================] - 2s 267us/step - loss: 0.5442 - acc: 0.7952
Epoch 2/100
8999/8999 [==============================] - 1s 131us/step - loss: 0.5202 - acc: 0.7963
Epoch 3/100
8999/8999 [==============================] - 1s 131us/step - loss: 0.5118 - acc: 0.7963
Epoch 4/100
8999/8999 [==============================] - 1s 124us/step - loss: 0.5065 - acc: 0.7963
Epoch 5/100
8999/8999 [==============================] - 1s 127us/step - loss: 0.5041 - acc: 0.7963
Epoch 6/100
8999/8999 [==============================] - 1s 128us/step - loss: 0.5041 - acc: 0.7963
Epoch 7/100
8999/8999 [==============================] - 1s 126us/step - loss: 0.5016 - acc: 0.7963
Epoch 8/100
8999/8999 [==============================] - 1s 129us/step - loss: 0.5018 - acc: 0.7963
Epoch 9/100
8999/8999 [==============================] - 1s 129us/step - loss: 0.5007 - acc: 0.7963
Epoch 10/100
8999/8999 [==============================] - 1s 119us/step - loss: 0.5020 - acc: 0.7963

8999/8999 [==============================] - 1s 114us/step - loss: 0.4978 - acc: 0.7963
Epoch 83/100
8999/8999 [==============================] - 1s 121us/step - loss: 0.4978 - acc: 0.7963
Epoch 84/100
8999/8999 [==============================] - 1s 118us/step - loss: 0.4981 - acc: 0.7963
Epoch 85/100
8999/8999 [==============================] - 1s 119us/step - loss: 0.4978 - acc: 0.7963
Epoch 86/100
8999/8999 [==============================] - 1s 119us/step - loss: 0.4979 - acc: 0.7963
Epoch 87/100
8999/8999 [==============================] - 1s 123us/step - loss: 0.4979 - acc: 0.7963
Epoch 88/100
8999/8999 [==============================] - 1s 120us/step - loss: 0.4979 - acc: 0.7963
Epoch 89/100
8999/8999 [==============================] - 1s 111us/step - loss: 0.4979 - acc: 0.7963
Epoch 90/100
8999/8999 [==============================] - 1s 111us/step - loss: 0.4979 - acc: 0.7963
Epoch 91/100
8999/8999 [==============================] - 1s 111us/step - loss: 0.4979 - acc: 0.7963
Epo

8999/8999 [==============================] - 1s 116us/step - loss: 0.4976 - acc: 0.7963
Epoch 64/100
8999/8999 [==============================] - 1s 110us/step - loss: 0.4976 - acc: 0.7963
Epoch 65/100
8999/8999 [==============================] - 1s 121us/step - loss: 0.4975 - acc: 0.7963
Epoch 66/100
8999/8999 [==============================] - 1s 121us/step - loss: 0.4976 - acc: 0.7963
Epoch 67/100
8999/8999 [==============================] - 1s 119us/step - loss: 0.4975 - acc: 0.7963
Epoch 68/100
8999/8999 [==============================] - 1s 120us/step - loss: 0.4975 - acc: 0.7963
Epoch 69/100
8999/8999 [==============================] - 1s 117us/step - loss: 0.4976 - acc: 0.7963
Epoch 70/100
8999/8999 [==============================] - 1s 117us/step - loss: 0.4975 - acc: 0.7963
Epoch 71/100
8999/8999 [==============================] - 1s 119us/step - loss: 0.4975 - acc: 0.7963
Epoch 72/100
8999/8999 [==============================] - 1s 113us/step - loss: 0.4975 - acc: 0.7963
Epo

8999/8999 [==============================] - 1s 121us/step - loss: 0.5001 - acc: 0.7963
Epoch 45/100
8999/8999 [==============================] - 1s 120us/step - loss: 0.4993 - acc: 0.7963
Epoch 46/100
8999/8999 [==============================] - 1s 123us/step - loss: 0.4993 - acc: 0.7963
Epoch 47/100
8999/8999 [==============================] - 1s 124us/step - loss: 0.4997 - acc: 0.7963
Epoch 48/100
8999/8999 [==============================] - 1s 110us/step - loss: 0.5004 - acc: 0.7963
Epoch 49/100
8999/8999 [==============================] - 1s 116us/step - loss: 0.4989 - acc: 0.7963
Epoch 50/100
8999/8999 [==============================] - 1s 121us/step - loss: 0.4986 - acc: 0.7963
Epoch 51/100
8999/8999 [==============================] - 1s 115us/step - loss: 0.4989 - acc: 0.7963
Epoch 52/100
8999/8999 [==============================] - 1s 115us/step - loss: 0.4986 - acc: 0.7963
Epoch 53/100
8999/8999 [==============================] - 1s 120us/step - loss: 0.4992 - acc: 0.7963
Epo

9000/9000 [==============================] - 1s 130us/step - loss: 0.4999 - acc: 0.7963 0s - loss: 0.508
Epoch 25/100
9000/9000 [==============================] - 1s 110us/step - loss: 0.5030 - acc: 0.7963
Epoch 26/100
9000/9000 [==============================] - 1s 107us/step - loss: 0.5046 - acc: 0.7963
Epoch 27/100
9000/9000 [==============================] - 1s 119us/step - loss: 0.5043 - acc: 0.7963
Epoch 28/100
9000/9000 [==============================] - 1s 123us/step - loss: 0.5034 - acc: 0.7963
Epoch 29/100
9000/9000 [==============================] - 1s 124us/step - loss: 0.5004 - acc: 0.7963
Epoch 30/100
9000/9000 [==============================] - 1s 111us/step - loss: 0.4995 - acc: 0.7963
Epoch 31/100
9000/9000 [==============================] - 1s 116us/step - loss: 0.4990 - acc: 0.7963
Epoch 32/100
9000/9000 [==============================] - 1s 122us/step - loss: 0.4988 - acc: 0.7963
Epoch 33/100
9000/9000 [==============================] - 1s 122us/step - loss: 0.4986 

...

9000/9000 [==============================] - 1s 114us/step - loss: 0.4973 - acc: 0.7963
Epoch 49/100
9000/9000 [==============================] - 1s 128us/step - loss: 0.4973 - acc: 0.7963
Epoch 50/100
9000/9000 [==============================] - 1s 127us/step - loss: 0.4972 - acc: 0.7963
Epoch 51/100
9000/9000 [==============================] - 1s 124us/step - loss: 0.4973 - acc: 0.7963
Epoch 52/100
9000/9000 [==============================] - 1s 115us/step - loss: 0.4973 - acc: 0.7963
Epoch 53/100
9000/9000 [==============================] - 1s 113us/step - loss: 0.4972 - acc: 0.7963
Epoch 54/100
9000/9000 [==============================] - 1s 116us/step - loss: 0.4974 - acc: 0.7963
Epoch 55/100
9000/9000 [==============================] - 1s 116us/step - loss: 0.4972 - acc: 0.7963
Epoch 56/100
9000/9000 [==============================] - 1s 129us/step - loss: 0.4973 - acc: 0.7963
Epoch 57/100
9000/9000 [==============================] - 1s 124us/step - loss: 0.4973 - acc: 0.7963
Epo

9001/9001 [==============================] - 1s 131us/step - loss: 0.5003 - acc: 0.7962
Epoch 30/100
9001/9001 [==============================] - 1s 131us/step - loss: 0.4998 - acc: 0.7962
Epoch 31/100
9001/9001 [==============================] - 1s 132us/step - loss: 0.5021 - acc: 0.7962
Epoch 32/100
9001/9001 [==============================] - 1s 121us/step - loss: 0.5011 - acc: 0.7962
Epoch 33/100
9001/9001 [==============================] - 1s 113us/step - loss: 0.5005 - acc: 0.7962
Epoch 34/100
9001/9001 [==============================] - 1s 114us/step - loss: 0.4999 - acc: 0.7962
Epoch 35/100
9001/9001 [==============================] - 1s 112us/step - loss: 0.5003 - acc: 0.7962
Epoch 36/100
9001/9001 [==============================] - 1s 112us/step - loss: 0.5017 - acc: 0.7962
Epoch 37/100
9001/9001 [==============================] - 1s 117us/step - loss: 0.4992 - acc: 0.7962
Epoch 38/100
9001/9001 [==============================] - 1s 125us/step - loss: 0.4996 - acc: 0.7962
Epo

9001/9001 [==============================] - 1s 136us/step - loss: 0.4999 - acc: 0.7962
Epoch 11/100
9001/9001 [==============================] - 1s 138us/step - loss: 0.5004 - acc: 0.7962
Epoch 12/100
9001/9001 [==============================] - 1s 130us/step - loss: 0.5006 - acc: 0.7962
Epoch 13/100
9001/9001 [==============================] - 1s 137us/step - loss: 0.5022 - acc: 0.7962
Epoch 14/100
9001/9001 [==============================] - 1s 140us/step - loss: 0.5004 - acc: 0.7962
Epoch 15/100
9001/9001 [==============================] - 1s 139us/step - loss: 0.4999 - acc: 0.7962
Epoch 16/100
9001/9001 [==============================] - 1s 136us/step - loss: 0.4995 - acc: 0.7962
Epoch 17/100
9001/9001 [==============================] - 1s 140us/step - loss: 0.5045 - acc: 0.7962
Epoch 18/100
9001/9001 [==============================] - 1s 139us/step - loss: 0.5013 - acc: 0.7962
Epoch 19/100
9001/9001 [==============================] - 1s 139us/step - loss: 0.5005 - acc: 0.7962
Epo

9001/9001 [==============================] - 1s 127us/step - loss: 0.4982 - acc: 0.7962
Epoch 92/100
9001/9001 [==============================] - 1s 127us/step - loss: 0.4982 - acc: 0.7962
Epoch 93/100
9001/9001 [==============================] - 1s 126us/step - loss: 0.4982 - acc: 0.7962
Epoch 94/100
9001/9001 [==============================] - 1s 131us/step - loss: 0.4982 - acc: 0.7962
Epoch 95/100
9001/9001 [==============================] - 1s 124us/step - loss: 0.4983 - acc: 0.7962
Epoch 96/100
9001/9001 [==============================] - 1s 126us/step - loss: 0.4983 - acc: 0.7962
Epoch 97/100
9001/9001 [==============================] - 1s 117us/step - loss: 0.4982 - acc: 0.7962
Epoch 98/100
9001/9001 [==============================] - 1s 121us/step - loss: 0.4982 - acc: 0.7962
Epoch 99/100
9001/9001 [==============================] - 1s 131us/step - loss: 0.4982 - acc: 0.7962
Epoch 100/100
9001/9001 [==============================] - 1s 120us/step - loss: 0.4983 - acc: 0.7962
ac

9001/9001 [==============================] - 1s 127us/step - loss: 0.4997 - acc: 0.7962
Epoch 73/100
9001/9001 [==============================] - 1s 124us/step - loss: 0.4987 - acc: 0.7962
Epoch 74/100
9001/9001 [==============================] - 1s 127us/step - loss: 0.5001 - acc: 0.7962
Epoch 75/100
9001/9001 [==============================] - 1s 132us/step - loss: 0.4987 - acc: 0.7962
Epoch 76/100
9001/9001 [==============================] - 1s 135us/step - loss: 0.5009 - acc: 0.7962
Epoch 77/100
9001/9001 [==============================] - 1s 119us/step - loss: 0.4995 - acc: 0.7962
Epoch 78/100
9001/9001 [==============================] - 1s 120us/step - loss: 0.4997 - acc: 0.7962
Epoch 79/100
9001/9001 [==============================] - 1s 137us/step - loss: 0.4991 - acc: 0.7962
Epoch 80/100
9001/9001 [==============================] - 1s 132us/step - loss: 0.4985 - acc: 0.7962
Epoch 81/100
9001/9001 [==============================] - 1s 122us/step - loss: 0.4986 - acc: 0.7962
Epo

In [74]:
cvscores

[79.62037962633413,
 79.62037963228865,
 79.62037962633413,
 79.5,
 79.60000000000001,
 79.60000000000001,
 79.5,
 79.47947948544592,
 79.67967968564611,
 79.57957958554601]